In [1]:
import pandas as pd

columns = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
    "wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised",
    "root_shell","su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count",
    "srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate",
    "same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate",
    "label","difficulty"
]

train_df = pd.read_csv("KDDTrain+.txt", names=columns)
test_df = pd.read_csv("KDDTest+.txt", names=columns)

print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)
print(train_df.head())
print(train_df["label"].value_counts())

Training shape: (125973, 43)
Test shape: (22544, 43)
   duration protocol_type   service flag  src_bytes  dst_bytes  land  \
0         0           tcp  ftp_data   SF        491          0     0   
1         0           udp     other   SF        146          0     0   
2         0           tcp   private   S0          0          0     0   
3         0           tcp      http   SF        232       8153     0   
4         0           tcp      http   SF        199        420     0   

   wrong_fragment  urgent  hot  ...  dst_host_same_srv_rate  \
0               0       0    0  ...                    0.17   
1               0       0    0  ...                    0.00   
2               0       0    0  ...                    0.10   
3               0       0    0  ...                    1.00   
4               0       0    0  ...                    1.00   

   dst_host_diff_srv_rate  dst_host_same_src_port_rate  \
0                    0.03                         0.17   
1                  

In [2]:
# Convert label to binary: normal vs attack
train_df['binary_label'] = train_df['label'].apply(lambda x: 'normal' if x == 'normal' else 'attack')
test_df['binary_label'] = test_df['label'].apply(lambda x: 'normal' if x == 'normal' else 'attack')

print(train_df['binary_label'].value_counts())
print(test_df['binary_label'].value_counts())

binary_label
normal    67343
attack    58630
Name: count, dtype: int64
binary_label
attack    12833
normal     9711
Name: count, dtype: int64


In [3]:
import numpy as np
from sklearn.preprocessing import LabelEncoder

categorical_cols = ['protocol_type', 'service', 'flag']

for col in categorical_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])
    # Handle labels in test set not seen in training set
    test_df[col] = test_df[col].map(lambda s: s if s in le.classes_ else 'unknown')
    le.classes_ = np.append(le.classes_, 'unknown')
    test_df[col] = le.transform(test_df[col])

print(train_df[categorical_cols].head())
print(test_df[categorical_cols].head())

   protocol_type  service  flag
0              1       20     9
1              2       44     9
2              1       49     5
3              1       24     9
4              1       24     9
   protocol_type  service  flag
0              1       49     1
1              1       49     1
2              1       20     9
3              0       14     9
4              1       60     2


In [4]:
from sklearn.preprocessing import StandardScaler

# Separate features (X) and target (y)
X_train = train_df.drop(columns=['label', 'difficulty', 'binary_label'])
y_train = train_df['binary_label']

X_test = test_df.drop(columns=['label', 'difficulty', 'binary_label'])
y_test = test_df['binary_label']

# Scale numeric features so large-range columns don't dominate
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train distribution:")
print(y_train.value_counts())

X_train shape: (125973, 41)
X_test shape: (22544, 41)
y_train distribution:
binary_label
normal    67343
attack    58630
Name: count, dtype: int64


In [5]:
# Convert target to numeric: normal=0, attack=1
y_train_num = y_train.map({'normal': 0, 'attack': 1})
y_test_num = y_test.map({'normal': 0, 'attack': 1})

print(y_train_num.value_counts())

binary_label
0    67343
1    58630
Name: count, dtype: int64


In [6]:
import pandas as pd

# Combine X_train with numeric target to compute correlation
corr_data = X_train.copy()
corr_data['target'] = y_train_num.values

correlations = corr_data.corr()['target'].drop('target').abs().sort_values(ascending=False)
print("Top 15 features by correlation with target:")
print(correlations.head(15))

Top 15 features by correlation with target:
same_srv_rate               0.751913
dst_host_srv_count          0.722535
dst_host_same_srv_rate      0.693803
logged_in                   0.690171
dst_host_srv_serror_rate    0.654985
dst_host_serror_rate        0.651842
serror_rate                 0.650652
srv_serror_rate             0.648289
flag                        0.647073
count                       0.576444
dst_host_count              0.375052
protocol_type               0.281355
service                     0.276548
srv_rerror_rate             0.253504
dst_host_srv_rerror_rate    0.253430
Name: target, dtype: float64


In [7]:
from sklearn.feature_selection import RFE
from sklearn.tree import DecisionTreeClassifier

estimator = DecisionTreeClassifier(random_state=42)
selector = RFE(estimator, n_features_to_select=15, step=1)
selector = selector.fit(X_train_scaled, y_train_num)

selected_features = X_train.columns[selector.support_]
print("Top 15 features selected by RFE (Wrapper Method):")
print(list(selected_features))

Top 15 features selected by RFE (Wrapper Method):
['duration', 'protocol_type', 'service', 'src_bytes', 'dst_bytes', 'hot', 'logged_in', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate']


In [8]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Use the RFE-selected features
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

# Train Decision Tree
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train_selected, y_train_num)
dt_preds = dt_model.predict(X_test_selected)

# Train Random Forest
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_selected, y_train_num)
rf_preds = rf_model.predict(X_test_selected)

# Compare accuracy
print("Decision Tree Accuracy:", accuracy_score(y_test_num, dt_preds))
print("Random Forest Accuracy:", accuracy_score(y_test_num, rf_preds))

Decision Tree Accuracy: 0.8272711142654364
Random Forest Accuracy: 0.7949787083037615


In [9]:
from sklearn.metrics import confusion_matrix, classification_report

print("=== Decision Tree ===")
print(confusion_matrix(y_test_num, dt_preds))
print(classification_report(y_test_num, dt_preds, target_names=['normal', 'attack']))

print("\n=== Random Forest ===")
print(confusion_matrix(y_test_num, rf_preds))
print(classification_report(y_test_num, rf_preds, target_names=['normal', 'attack']))

=== Decision Tree ===
[[9319  392]
 [3502 9331]]
              precision    recall  f1-score   support

      normal       0.73      0.96      0.83      9711
      attack       0.96      0.73      0.83     12833

    accuracy                           0.83     22544
   macro avg       0.84      0.84      0.83     22544
weighted avg       0.86      0.83      0.83     22544


=== Random Forest ===
[[9425  286]
 [4336 8497]]
              precision    recall  f1-score   support

      normal       0.68      0.97      0.80      9711
      attack       0.97      0.66      0.79     12833

    accuracy                           0.79     22544
   macro avg       0.83      0.82      0.79     22544
weighted avg       0.85      0.79      0.79     22544



In [10]:
import joblib

# Save the Decision Tree model (best recall on attacks — our recommended model)
joblib.dump(dt_model, 'intrusion_model.pkl')

# Save separate encoders for protocol_type and service (needed for the app)
from sklearn.preprocessing import LabelEncoder

protocol_encoder = LabelEncoder()
protocol_encoder.fit(['tcp', 'udp', 'icmp'])
joblib.dump(protocol_encoder, 'protocol_encoder.pkl')

service_encoder = LabelEncoder()
service_encoder.fit(train_df['service'].astype(str))  # this may show numbers since we already encoded it — that's fine
joblib.dump(service_encoder, 'service_encoder.pkl')

# Save the exact list of features the model expects, in order
joblib.dump(list(selected_features), 'selected_features.pkl')

print("Saved: intrusion_model.pkl, protocol_encoder.pkl, service_encoder.pkl, selected_features.pkl")

Saved: intrusion_model.pkl, protocol_encoder.pkl, service_encoder.pkl, selected_features.pkl


In [13]:
import joblib

# Grab one real Normal example and one real Attack example from the test set
normal_idx = y_test_num[y_test_num == 0].index[0]
attack_idx = y_test_num[y_test_num == 1].index[0]

normal_example = X_test.loc[normal_idx, selected_features].to_dict()
attack_example = X_test.loc[attack_idx, selected_features].to_dict()

examples = {'normal': normal_example, 'attack': attack_example}
joblib.dump(examples, 'examples.pkl')

print("Saved examples.pkl")
print("Normal example:", normal_example)
print("Attack example:", attack_example)

Saved examples.pkl
Normal example: {'duration': 2.0, 'protocol_type': 1.0, 'service': 20.0, 'src_bytes': 12983.0, 'dst_bytes': 0.0, 'hot': 0.0, 'logged_in': 0.0, 'dst_host_count': 134.0, 'dst_host_srv_count': 86.0, 'dst_host_same_srv_rate': 0.61, 'dst_host_same_src_port_rate': 0.61, 'dst_host_srv_diff_host_rate': 0.02, 'dst_host_serror_rate': 0.0, 'dst_host_rerror_rate': 0.0, 'dst_host_srv_rerror_rate': 0.0}
Attack example: {'duration': 0.0, 'protocol_type': 1.0, 'service': 49.0, 'src_bytes': 0.0, 'dst_bytes': 0.0, 'hot': 0.0, 'logged_in': 0.0, 'dst_host_count': 255.0, 'dst_host_srv_count': 10.0, 'dst_host_same_srv_rate': 0.04, 'dst_host_same_src_port_rate': 0.0, 'dst_host_srv_diff_host_rate': 0.0, 'dst_host_serror_rate': 0.0, 'dst_host_rerror_rate': 1.0, 'dst_host_srv_rerror_rate': 1.0}
